# 43 — Preparation Protocols

**Repo:** `decoherence-free-sensing`  
**Notebook role:** experimental-readiness layer  
**Previous:** `37_two_body_measurements.ipynb`  
**Next:** hardware-specific implementation / optimization

Notebook 43 closes the first engineering loop.

Previous notebooks specified the DFS constraint, the differential generator, admissible Lieb-Mattis states, Quantum Fisher Information, and two-body measurement readout.

Notebook 43 asks the final qualification question:

> How do we know the laboratory has actually prepared a useful DFS sensing resource?

Preparation specifies experimental readiness.


## Engineering statement

A preparation protocol is admissible only if it produces:

1. high DFS support,
2. high target fidelity,
3. retained differential sensitivity.

DFS support specifies admissibility.  
Target fidelity specifies correctness.  
Differential sensitivity specifies usefulness.


## Key equations

DFS projector:

\[
P_{\rm DFS}
=
\sum_{\psi\in J_z^+=0}
|\psi\rangle\langle\psi|.
\]

DFS support:

\[
p_{\rm DFS}
=
\langle\psi_{\rm prep}|P_{\rm DFS}|\psi_{\rm prep}\rangle.
\]

Target fidelity:

\[
\mathcal F
=
\left|
\langle
\psi_{\rm LM}
|
\psi_{\rm prep}
\rangle
\right|^2.
\]

Retained differential sensitivity:

\[
\mathrm{Var}_{\psi_{\rm prep}}(J_z^-)>0.
\]

Quantum Fisher proxy:

\[
F_Q
=
4\,\mathrm{Var}_{\psi_{\rm prep}}(J_z^-).
\]

Readiness condition:

\[
p_{\rm DFS}\approx 1,
\qquad
\mathcal F\approx 1,
\qquad
F_Q>0.
\]


## Notebook outputs

Running this notebook creates:

- `results/figures/43_preparation_protocol_map.png`
- `results/figures/43_target_vs_prepared_state.png`
- `results/figures/43_dfs_support.png`
- `results/figures/43_preparation_quality_dashboard.png`
- `results/figures/43_preparation_specification.png`
- `results/preparation_protocol_summary.csv`
- `results/preparation_protocol_summary.json`
- `results/decoherence_free_sensing_preparation_protocols_outputs.zip`


In [ ]:
from pathlib import Path
import json
import zipfile
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

ROOT = Path.cwd()
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"

for path in [RESULTS, FIGURES]:
    path.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 220,
    "font.size": 13,
    "axes.titlesize": 21,
    "axes.labelsize": 14,
    "legend.fontsize": 11,
})

def savefig(name):
    path = FIGURES / name
    plt.savefig(path, bbox_inches="tight")
    return path

def draw_box(ax, xy, width, height, title, body=None, fontsize=14, facecolor="white", lw=1.8):
    x, y = xy
    box = FancyBboxPatch(
        (x, y), width, height,
        boxstyle="round,pad=0.025,rounding_size=0.04",
        linewidth=lw,
        edgecolor="black",
        facecolor=facecolor,
    )
    ax.add_patch(box)
    ax.text(
        x + width/2,
        y + height*0.64 if body else y + height/2,
        title,
        ha="center",
        va="center",
        fontsize=fontsize,
        fontweight="bold"
    )
    if body:
        ax.text(
            x + width/2,
            y + height*0.32,
            body,
            ha="center",
            va="center",
            fontsize=11
        )

def draw_arrow(ax, p0, p1):
    ax.add_patch(FancyArrowPatch(
        p0, p1,
        arrowstyle="-|>",
        mutation_scale=18,
        linewidth=1.8
    ))


## 1. Preparation protocol map

The preparation notebook turns the target-state specification into an experimental qualification workflow.

Three broad routes are represented:

- unitary preparation,
- dissipative preparation,
- measurement-conditioned preparation.

Each route is judged by the same three readiness checks:

\[
p_{\rm DFS},
\qquad
\mathcal F,
\qquad
F_Q.
\]

The figure uses plain text labels for Matplotlib compatibility; the mathematical notation remains in the notebook text.


In [ ]:
fig, ax = plt.subplots(figsize=(13.4, 6.4))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.93, "Preparation Protocol Map", ha="center", va="center", fontsize=23)

draw_box(ax, (0.05, 0.48), 0.18, 0.28, "Ideal target", "psi_LM")
draw_box(ax, (0.31, 0.62), 0.18, 0.20, "Unitary", "control\nsequence")
draw_box(ax, (0.31, 0.40), 0.18, 0.20, "Dissipative", "engineered\nrelaxation")
draw_box(ax, (0.31, 0.18), 0.18, 0.20, "Conditioned", "measurement\nfeedback")
draw_box(ax, (0.59, 0.44), 0.18, 0.30, "Prepared state", "psi_prep")
draw_box(ax, (0.82, 0.44), 0.14, 0.30, "Verify", "p_DFS\nFidelity\nF_Q")

draw_arrow(ax, (0.23, 0.62), (0.30, 0.72))
draw_arrow(ax, (0.23, 0.62), (0.30, 0.50))
draw_arrow(ax, (0.23, 0.62), (0.30, 0.28))
draw_arrow(ax, (0.49, 0.72), (0.58, 0.60))
draw_arrow(ax, (0.49, 0.50), (0.58, 0.58))
draw_arrow(ax, (0.49, 0.28), (0.58, 0.52))
draw_arrow(ax, (0.77, 0.59), (0.81, 0.59))

ax.text(
    0.5, 0.08,
    "Experimental preparation specifies whether the sensing resource exists.",
    ha="center", va="center", fontsize=14
)

path = savefig("43_preparation_protocol_map.png")
plt.show()
path


## 2. Target versus prepared state

A useful preparation protocol should produce a state close to the target Lieb-Mattis state.

We model this with a compact amplitude proxy:

\[
|\psi_{\rm LM}\rangle
=
\sum_x (-1)^x c_x |x\rangle_A|\bar{x}\rangle_B.
\]

A prepared state may have imperfect amplitudes, phase noise, and small leakage outside the central DFS sector.


In [ ]:
rng = np.random.default_rng(43)

n = 10
x = np.arange(n + 1)
weights = np.array([math.comb(n, int(k)) for k in x], dtype=float)
c = np.sqrt(weights)
c = c / np.linalg.norm(c)
target_amp = ((-1) ** x) * c

amp_noise = rng.normal(0, 0.035, len(x))
prepared_amp = target_amp + amp_noise
prepared_amp = prepared_amp / np.linalg.norm(prepared_amp)

fidelity = float(np.abs(np.vdot(target_amp, prepared_amp)) ** 2)

fig, ax = plt.subplots(figsize=(9.0, 5.8))
bar_w = 0.36
ax.bar(x - bar_w/2, target_amp, width=bar_w, alpha=0.8, label="target Lieb-Mattis")
ax.bar(x + bar_w/2, prepared_amp, width=bar_w, alpha=0.7, label="prepared proxy")
ax.axhline(0, linewidth=1.0)
ax.set_title("Target vs Prepared State")
ax.set_xlabel("central DFS basis index x")
ax.set_ylabel("signed amplitude")
ax.grid(True, axis="y", alpha=0.25)
ax.legend()

ax.text(
    0.5, 0.86,
    f"Fidelity ≈ {fidelity:.3f}",
    transform=ax.transAxes,
    ha="center",
    va="center",
    fontsize=14,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.9)
)

path = savefig("43_target_vs_prepared_state.png")
plt.show()
path


## 3. DFS support

Fidelity alone is not enough.

A state can resemble the target within part of the basis while still leaking outside the protected sector. The DFS support is

\[
p_{\rm DFS}
=
\langle\psi_{\rm prep}|P_{\rm DFS}|\psi_{\rm prep}\rangle.
\]

High support means the prepared state remains admissible.


In [ ]:
states = ["ideal", "prepared", "leaky"]
dfs_support = np.array([1.00, 0.94, 0.58])
leakage = 1 - dfs_support

fig, ax = plt.subplots(figsize=(8.4, 5.8))
ax.bar(states, dfs_support, alpha=0.82, label="p_DFS")
ax.bar(states, leakage, bottom=dfs_support, alpha=0.35, label="leakage")
ax.axhline(0.9, linestyle="--", linewidth=1.5, label="readiness threshold")
ax.set_ylim(0, 1.08)
ax.set_title("DFS Support Qualifies Admissibility")
ax.set_ylabel("population fraction")
ax.grid(True, axis="y", alpha=0.25)
ax.legend(loc="lower left")

for i, val in enumerate(dfs_support):
    ax.text(i, val - 0.08, f"{val:.2f}", ha="center", va="center", fontsize=13)

path = savefig("43_dfs_support.png")
plt.show()
path


## 4. Preparation quality dashboard

Preparation readiness is multi-constraint.

A prepared state must be:

- inside the DFS,
- close to the target state,
- still sensitive to \(J_z^-\).

The dashboard collects these checks into one qualification view.


In [ ]:
metrics = {
    "DFS support": 0.94,
    "Target fidelity": fidelity,
    "Differential sensitivity": 0.88,
}
threshold = 0.80

fig, ax = plt.subplots(figsize=(10.0, 5.8))
names = list(metrics.keys())
values = np.array(list(metrics.values()))

ax.barh(names, values, alpha=0.82)
ax.axvline(threshold, linestyle="--", linewidth=1.6, label="ready threshold")
ax.set_xlim(0, 1.05)
ax.set_title("Preparation Quality Dashboard")
ax.set_xlabel("score")
ax.grid(True, axis="x", alpha=0.25)
ax.legend(loc="lower right")

for i, val in enumerate(values):
    label = "ready" if val >= threshold else "refine"
    ax.text(val + 0.025, i, f"{val:.2f}  {label}", va="center", fontsize=12)

path = savefig("43_preparation_quality_dashboard.png")
plt.show()
path


## 5. Preparation specification

The preparation specification is:

\[
\text{Target}
\rightarrow
\text{Prepare}
\rightarrow
\text{Verify DFS}
\rightarrow
\text{Verify sensitivity}
\rightarrow
\text{Measure}.
\]

The point is not just to create entanglement. The point is to create an entangled state that remains admissible and useful for the measurement protocol.


In [ ]:
fig, ax = plt.subplots(figsize=(13.2, 5.8))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.92, "Preparation Specification", ha="center", va="center", fontsize=23)

items = [
    (0.05, "Target", "psi_LM"),
    (0.245, "Prepare", "unitary / dissipative\nconditioned"),
    (0.44, "Verify", "p_DFS ≈ 1"),
    (0.635, "Qualify", "Fidelity ≈ 1\nF_Q > 0"),
    (0.83, "Measure", "two-body\nreadout"),
]
w, h, y = 0.15, 0.32, 0.42

for x0, title, body in items:
    draw_box(ax, (x0, y), w, h, title, body)

for x0 in [0.20, 0.395, 0.59, 0.785]:
    draw_arrow(ax, (x0, y + h/2), (x0 + 0.04, y + h/2))

ax.text(
    0.5, 0.20,
    "Preparation specifies sensing readiness.",
    ha="center", va="center", fontsize=14
)

path = savefig("43_preparation_specification.png")
plt.show()
path


## 6. Readiness table


In [ ]:
jz_minus = 2 * x - n
prob_prep = np.abs(prepared_amp) ** 2
mean_minus = float(np.sum(prob_prep * jz_minus))
var_minus = float(np.sum(prob_prep * (jz_minus - mean_minus) ** 2))
fq_proxy = 4 * var_minus

summary = pd.DataFrame([
    {
        "quantity": "P_DFS",
        "meaning": "projector onto central DFS sector",
        "target": "defined by J_z^+ = 0",
        "score": ""
    },
    {
        "quantity": "p_DFS",
        "meaning": "DFS support of prepared state",
        "target": "maximize",
        "score": f"{dfs_support[1]:.3f}"
    },
    {
        "quantity": "Fidelity",
        "meaning": "overlap with Lieb-Mattis target",
        "target": "maximize",
        "score": f"{fidelity:.3f}"
    },
    {
        "quantity": "Var(J_z^-)",
        "meaning": "retained differential sensitivity",
        "target": "maximize",
        "score": f"{var_minus:.3f}"
    },
    {
        "quantity": "F_Q proxy",
        "meaning": "4 Var(J_z^-)",
        "target": "maximize",
        "score": f"{fq_proxy:.3f}"
    },
    {
        "quantity": "leakage",
        "meaning": "population outside DFS",
        "target": "minimize",
        "score": f"{1-dfs_support[1]:.3f}"
    },
])

summary_csv = RESULTS / "preparation_protocol_summary.csv"
summary_json = RESULTS / "preparation_protocol_summary.json"

summary.to_csv(summary_csv, index=False)
summary.to_json(summary_json, orient="records", indent=2)

summary


## 7. Export notebook outputs

Run the next cell after generating figures.

It creates a zip archive with all Notebook 43 figures and readiness summaries.


In [ ]:
zip_path = RESULTS / "decoherence_free_sensing_preparation_protocols_outputs.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for png in sorted(FIGURES.glob("43_*.png")):
        zf.write(png, png.relative_to(ROOT))
    zf.write(summary_csv, summary_csv.relative_to(ROOT))
    zf.write(summary_json, summary_json.relative_to(ROOT))

print(f"Created {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))

zip_path


## 8. Completion

Notebook 43 closes the first repository arc:

\[
\text{DFS constraint}
\rightarrow
\text{admissible state}
\rightarrow
\text{QFI}
\rightarrow
\text{measurement}
\rightarrow
\text{preparation readiness}.
\]

The next notebooks can become hardware-specific:

- noise channels,
- finite control error,
- detector readout,
- cavity or trapped-ion implementation,
- preparation optimization.
